<!--nav--> [🗺 Learning path](README.md) · **29/44** · ◀ [Reading the Logs](./Serving_Logs_Observability.ipynb) · [Structured Output & Guided Decoding](./Structured_Output_Guided_Decoding.ipynb) ▶

# Capstone: Benchmarking, SLOs & Capacity Planning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sugeerth/gpu-training-notebooks/blob/main/Serving_Benchmark_Capacity_Planning.ipynb)

You can now explain, visualize, and monitor a serving stack. The last question is the one your boss
actually asks: **"how many GPUs do we need, and what will it cost?"**

This notebook answers it with numbers you generate yourself.

| Part | What you'll do |
|---|---|
| **1** | Learn why most LLM benchmarks are wrong (closed-loop testing, means instead of tails) |
| **2** | Build an **open-loop Poisson load generator** — offline simulator (CPU) *and* a live version for a real server |
| **3** | Sweep load to find **the knee** — the point where latency explodes, and why Little's Law predicts it |
| **4** | Define an SLO and measure **goodput** — the only capacity number that means anything |
| **5** | Compute **$ per million tokens** across GPUs, and the break-even for quantization/speculation |
| **6** | The **decision tree**: symptom → which earlier notebook to reach for |

**Runs on:** any CPU. The simulator is calibrated against a roofline model of real GPU behavior; a
GPU-gated cell at the end runs the identical load generator against a live vLLM server.

In [ ]:
import math, random, json, statistics, uuid
from collections import defaultdict
from IPython.display import HTML, display

D3_URL = "https://cdn.jsdelivr.net/npm/d3@7/dist/d3.min.js"

def show_d3(js, data=None, height=420):
    div = f"viz_{uuid.uuid4().hex[:10]}"
    html = f'''
<div id="{div}" style="width:100%;max-width:920px;font-family:system-ui,sans-serif"></div>
<script>
(function() {{
  function run() {{
    const d3 = window.d3;
    const root = d3.select("#{div}");
    const data = {json.dumps(data)};
    const W = (document.getElementById("{div}").clientWidth || 880), H = {height};
    try {{ {js} }} catch (e) {{ root.append("pre").style("color","crimson").text("viz error: " + e); }}
  }}
  if (window.d3) run();
  else {{
    const s = document.createElement("script");
    s.src = "{D3_URL}"; s.onload = run;
    s.onerror = () => document.getElementById("{div}").textContent = "Could not load D3 from the CDN.";
    document.head.appendChild(s);
  }}
}})();
</script>'''
    display(HTML(html))
print("ready")

## Part 1 · Why most LLM benchmarks lie

**Mistake 1 — closed-loop load testing.** The `ThreadPoolExecutor(max_workers=32)` pattern from
[vLLM High-Throughput Serving](./vLLM_High_Throughput_Serving.ipynb) (and most blog posts) keeps *exactly* 32 requests in flight: a new one starts only when
an old one finishes. Real users don't work that way — they arrive on their own schedule, and if
your server slows down, **more** of them pile up, not fewer. Closed-loop testing has a built-in
safety valve that hides the exact failure you're trying to find.

```
CLOSED loop (what most benchmarks do)      OPEN loop (what production is)
  ┌───────────────┐                          arrivals ~ Poisson(λ), independent of server state
  │ 32 workers    │ ← server slows,             ↓ ↓  ↓    ↓ ↓ ↓ ↓ ↓ ↓ ↓ ↓ ↓ ↓ ↓ ↓
  │ loop forever  │   arrivals slow too      [ queue grows without bound if λ > capacity ]
  └───────────────┘   (self-throttling!)         → this is where the knee lives
```

**Mistake 2 — reporting the mean.** Mean TTFT is dominated by the easy requests. Your users notice
p95 and p99. ([Reading the Logs](./Serving_Logs_Observability.ipynb) showed a mean of ~370ms hiding a p95 of ~1.1s.)

**Mistake 3 — no warmup.** The first requests pay CUDA graph capture, autotuning, and cold prefix
caches. Always discard them.

**Mistake 4 — one prompt shape.** A benchmark of 100-token prompts tells you nothing about a RAG
workload with 4k-token prompts. Prefill and decode scale differently ([Serving Fundamentals](./Serving_Fundamentals_KV_Cache_Batching.ipynb)) — measure *your*
distribution.

Everything below is open-loop, percentile-reported, warmed up, and parameterized by prompt shape.

## Part 2 · The simulator (and why it's trustworthy)

We model the engine with a **roofline**: total decode throughput rises with batch size until it hits
the memory-bandwidth ceiling, then flattens. That single curve is the behavior you measured for real
in [Serving Fundamentals](./Serving_Fundamentals_KV_Cache_Batching.ipynb)'s batch-scaling cell.

```
total decode tok/s = min(batch × per_request_rate , ceiling)
per-request rate   = total / batch          ← so TPOT degrades once you're on the ceiling
```

Admission is limited by **both** `max_num_seqs` (slots) and the **KV token pool** — the two limits
from [Reading the Logs](./Serving_Logs_Observability.ipynb)'s startup log.

In [ ]:
class ServerModel:
    '''A roofline model of one GPU running vLLM. Defaults ≈ Qwen2.5-0.5B on a T4 (serving-fundamentals, logs).'''
    def __init__(self, kv_tokens=371_264, max_num_seqs=32,
                 per_req_decode_tps=24.0, decode_ceiling_tps=620.0,
                 prefill_tps=9000.0):
        self.kv_tokens, self.max_num_seqs = kv_tokens, max_num_seqs
        self.per_req, self.ceiling, self.prefill_tps = per_req_decode_tps, decode_ceiling_tps, prefill_tps

    def decode_rate_per_request(self, batch):
        if batch == 0: return 0.0
        return min(self.per_req, self.ceiling / batch)      # the roofline, per request

def simulate(model, arrival_rate, seconds=400, prompt_mean=800, prompt_sd=250,
             output_mean=200, output_sd=90, dt=0.05, warmup=60, seed=0):
    '''Open-loop discrete-time simulation. Returns per-request latency records.'''
    rng = random.Random(seed)
    queue, running, records = [], [], []
    kv_used = 0.0
    steps = int(seconds / dt)
    for step in range(steps):
        now = step * dt
        # --- Poisson arrivals ---
        for _ in range(rng.poisson(arrival_rate * dt) if hasattr(rng, "poisson")
                       else _poisson(rng, arrival_rate * dt)):
            queue.append({"born": now,
                          "prompt": max(16, int(rng.gauss(prompt_mean, prompt_sd))),
                          "need": max(8, int(rng.gauss(output_mean, output_sd))),
                          "done": 0, "first": None})
        # --- admission: needs a slot AND enough KV ---
        while queue and len(running) < model.max_num_seqs:
            r = queue[0]
            if kv_used + r["prompt"] + r["need"] > model.kv_tokens:
                break
            kv_used += r["prompt"] + r["need"]
            r["admitted"] = now
            r["prefill_done"] = now + r["prompt"] / model.prefill_tps
            running.append(queue.pop(0))
        # --- decode step ---
        rate = model.decode_rate_per_request(len(running))
        for r in list(running):
            if now < r["prefill_done"]:
                continue                                  # still prefilling
            if r["first"] is None:
                r["first"] = now                          # first token emitted
            r["done"] += rate * dt
            if r["done"] >= r["need"]:
                kv_used -= r["prompt"] + r["need"]
                running.remove(r)
                if r["born"] >= warmup:                   # discard warmup
                    records.append({
                        "ttft": r["first"] - r["born"],
                        "tpot": (now - r["first"]) / max(r["need"], 1),
                        "e2e": now - r["born"],
                        "queue_time": r["admitted"] - r["born"],
                        "tokens": r["need"], "prompt": r["prompt"]})
    dur = seconds - warmup
    return {"records": records, "arrival_rate": arrival_rate,
            "completed_per_s": len(records) / dur,
            "out_tokens_per_s": sum(r["tokens"] for r in records) / dur,
            "unfinished": len(running) + len(queue)}

def _poisson(rng, lam):                                   # Knuth, for stdlib Random
    L, k, p = math.exp(-lam), 0, 1.0
    while True:
        p *= rng.random()
        if p <= L: return k
        k += 1

def pct(vals, q):
    if not vals: return float("nan")
    s = sorted(vals); i = min(len(s) - 1, int(q * len(s)))
    return s[i]

model = ServerModel()
r = simulate(model, arrival_rate=1.5)
print(f"λ=1.5 req/s -> completed {r['completed_per_s']:.2f} req/s, "
      f"{r['out_tokens_per_s']:.0f} output tok/s")
print(f"  TTFT  p50 {pct([x['ttft'] for x in r['records']], .5)*1000:6.0f}ms  "
      f"p95 {pct([x['ttft'] for x in r['records']], .95)*1000:6.0f}ms")
print(f"  TPOT  p50 {pct([x['tpot'] for x in r['records']], .5)*1000:6.1f}ms  "
      f"p95 {pct([x['tpot'] for x in r['records']], .95)*1000:6.1f}ms")
print(f"  E2E   p50 {pct([x['e2e'] for x in r['records']], .5):6.2f}s   "
      f"p95 {pct([x['e2e'] for x in r['records']], .95):6.2f}s")

## Part 3 · Find the knee

Now sweep the arrival rate and watch what happens. **Throughput rises, then flattens. Latency is
flat, then explodes.** The explosion point is the knee — and it is *not* where your GPU utilization
hits 100%; it's earlier, because queueing theory is unkind.

Little's Law explains it in one line:

$$L = \lambda W$$

*(requests in the system) = (arrival rate) × (time each spends in the system)*. Once λ approaches
the service capacity μ, W grows without bound — for M/M/1, `W = 1/(μ−λ)`. That `μ−λ` in the
denominator is the entire story of capacity planning: at 90% utilization your wait is 10× the
service time; at 99% it's 100×.

In [ ]:
SLO_TTFT, SLO_TPOT = 1.0, 0.05        # 1s to first token, 50ms/token (~20 tok/s streaming)

sweep = []
for lam in [0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0, 2.25, 2.5, 2.75, 3.0, 3.5, 4.0]:
    s = simulate(model, arrival_rate=lam, seconds=360, seed=1)
    recs = s["records"]
    good = [x for x in recs if x["ttft"] <= SLO_TTFT and x["tpot"] <= SLO_TPOT]
    dur = 360 - 60
    sweep.append({
        "lam": lam,
        "throughput": s["completed_per_s"],
        "out_tps": s["out_tokens_per_s"],
        "ttft_p50": pct([x["ttft"] for x in recs], .50),
        "ttft_p95": pct([x["ttft"] for x in recs], .95),
        "tpot_p95": pct([x["tpot"] for x in recs], .95),
        "e2e_p95":  pct([x["e2e"]  for x in recs], .95),
        "goodput":  len(good) / dur,
        "slo_ok":   len(good) / max(len(recs), 1),
        "backlog":  s["unfinished"]})

print(f"{'λ req/s':>8}{'done/s':>8}{'out tok/s':>11}{'TTFT p95':>10}{'TPOT p95':>10}{'SLO met':>9}{'goodput':>9}")
print("-" * 65)
for row in sweep:
    flag = "  <= knee" if 0 < row["slo_ok"] < 0.98 and row["slo_ok"] > 0.5 else ""
    print(f"{row['lam']:>8.2f}{row['throughput']:>8.2f}{row['out_tps']:>11.0f}"
          f"{row['ttft_p95']*1000:>9.0f}ms{row['tpot_p95']*1000:>9.1f}ms"
          f"{row['slo_ok']:>8.0%}{row['goodput']:>9.2f}{flag}")

best = max(sweep, key=lambda r: r["goodput"])
print(f"\nPeak goodput {best['goodput']:.2f} req/s at λ={best['lam']} "
      f"({best['slo_ok']:.0%} of requests met the SLO)")
print("Push past that and raw throughput barely moves while p95 latency runs away —")
print("you would be serving MORE requests badly instead of fewer requests well.")

In [ ]:
# The knee, drawn. Drag the SLO sliders and watch the usable-capacity line move.
JS = r'''
const M = {top: 20, right: 60, bottom: 44, left: 58};
const iw = W - M.left - M.right, ih = H - M.top - M.bottom;

const ctr = root.append("div").style("font-size", "13px").style("margin-bottom", "6px");
function slider(label, min, max, val, step, fmt) {
  const w = ctr.append("label").style("margin-right", "20px");
  w.append("span").text(label + " ");
  const out = w.append("b");
  const inp = w.append("input").attr("type", "range").attr("min", min).attr("max", max)
      .attr("step", step).attr("value", val).style("vertical-align", "middle").style("margin-left", "6px");
  inp.on("input", () => { out.text(fmt(+inp.node().value)); draw(); });
  out.text(fmt(val));
  return inp;
}
const sTtft = slider("TTFT SLO:", 0.2, 3, 1.0, 0.1, v => v.toFixed(1) + "s");

const svg = root.append("svg").attr("width", W).attr("height", H)
    .append("g").attr("transform", `translate(${M.left},${M.top})`);
const x = d3.scaleLinear().domain(d3.extent(data, d => d.lam)).range([0, iw]);
const yL = d3.scaleLog().domain([0.02, d3.max(data, d => d.ttft_p95) * 1.3]).range([ih, 0]);
const yT = d3.scaleLinear().domain([0, d3.max(data, d => d.throughput) * 1.25]).range([ih, 0]);

svg.append("g").attr("transform", `translate(0,${ih})`).call(d3.axisBottom(x));
svg.append("g").call(d3.axisLeft(yL).ticks(5, "~s"));
svg.append("g").attr("transform", `translate(${iw},0)`).call(d3.axisRight(yT).ticks(5));
svg.append("text").attr("x", iw/2).attr("y", ih+36).attr("text-anchor","middle")
   .style("font-size","12px").text("offered load λ (requests/s)");
svg.append("text").attr("transform","rotate(-90)").attr("x",-ih/2).attr("y",-42)
   .attr("text-anchor","middle").style("font-size","12px").style("fill","#d32f2f")
   .text("p95 latency (s, log)");
svg.append("text").attr("transform","rotate(90)").attr("x",ih/2).attr("y",-iw-42)
   .attr("text-anchor","middle").style("font-size","12px").style("fill","#1976d2")
   .text("throughput / goodput (req/s)");

const line = (yf, acc) => d3.line().x(d => x(d.lam)).y(d => yf(acc(d)));
svg.append("path").datum(data).attr("fill","none").attr("stroke","#d32f2f").attr("stroke-width",2)
   .attr("d", line(yL, d => d.ttft_p95));
svg.append("path").datum(data).attr("fill","none").attr("stroke","#1976d2").attr("stroke-width",2)
   .attr("d", line(yT, d => d.throughput));
const goodPath = svg.append("path").attr("fill","none").attr("stroke","#2e7d32")
   .attr("stroke-width",2).attr("stroke-dasharray","5 3");
const sloLine = svg.append("line").attr("stroke","#d32f2f").attr("stroke-dasharray","4 3").attr("x1",0).attr("x2",iw);
const kneeLine = svg.append("line").attr("y1",0).attr("y2",ih).attr("stroke","#333").attr("stroke-width",1.5);
const kneeTxt = svg.append("text").style("font-size","12px").style("font-weight",700);
svg.selectAll("dot").data(data).join("circle").attr("cx",d=>x(d.lam)).attr("cy",d=>yL(d.ttft_p95))
   .attr("r",3).attr("fill","#d32f2f").append("title").text(d=>`λ=${d.lam}: p95 ${d.ttft_p95.toFixed(2)}s`);
svg.append("text").attr("x", 6).attr("y", 12).style("font-size","11.5px")
   .html("").append("tspan").style("fill","#d32f2f").text("red = p95 TTFT");
svg.append("text").attr("x", 96).attr("y", 12).style("font-size","11.5px").style("fill","#1976d2").text("blue = throughput");
svg.append("text").attr("x", 208).attr("y", 12).style("font-size","11.5px").style("fill","#2e7d32").text("green dashed = goodput (SLO-meeting)");

function draw() {
  const slo = +sTtft.node().value;
  const rows = data.map(d => ({...d, ok: d.ttft_p95 <= slo}));
  goodPath.datum(rows).attr("d", line(yT, d => d.ok ? d.throughput : d.throughput * d.slo_ok));
  sloLine.attr("y1", yL(slo)).attr("y2", yL(slo));
  const knee = rows.filter(d => d.ok).slice(-1)[0] || rows[0];
  kneeLine.attr("x1", x(knee.lam)).attr("x2", x(knee.lam));
  kneeTxt.attr("x", Math.min(x(knee.lam) + 6, iw - 150)).attr("y", 34)
      .text(`usable capacity ≈ ${knee.lam} req/s`);
}
draw();
'''
show_d3(JS, sweep, height=380)

**How to read it.** The blue throughput curve flattens — that's your GPU saturating. The red p95
curve is flat and then vertical — that's the queue. **The gap between where blue flattens and where
red explodes is your safety margin**, and it's why you provision at ~70% of measured peak, not 95%.

Drag the TTFT slider: a stricter SLO doesn't just change a number on a dashboard, it *reduces the
load one GPU can carry*. Capacity is a function of your promise to users.

## Part 4 · Goodput: the only number worth reporting

**Throughput** counts requests you served. **Goodput** counts requests you served *acceptably*.
Past the knee they diverge violently — you can double your "throughput" while serving everyone
badly. Notice in the table above how goodput peaks and then *falls* while throughput still rises.

That peak is the number to put in a capacity plan:

```
GPUs needed = ceil( peak_traffic_req_per_s / goodput_per_gpu ) × (1 + headroom)
```

with headroom ≥ 30% for traffic spikes, deploys, and the failure of one replica.

## Part 5 · What does a million tokens cost?

Everything so far becomes money with one division:

```
$ per 1M output tokens = (GPU $/hour ÷ 3600) ÷ (output tokens/s) × 1,000,000
```

Prices below are **illustrative 2025-ish on-demand rates** — they vary hugely by cloud, region, and
commitment, so edit them. What's durable is the *shape* of the conclusion.

In [ ]:
# Illustrative on-demand prices (USD/hr) and the throughput each GPU sustains for our 0.5B model.
# The tok/s numbers scale roughly with memory bandwidth - the bottleneck from the serving-fundamentals notebook.
GPUS = [
    {"gpu": "T4 16GB",    "usd_hr": 0.35, "bw_gbs": 300,  "out_tps": 620},
    {"gpu": "L4 24GB",    "usd_hr": 0.70, "bw_gbs": 300,  "out_tps": 700},
    {"gpu": "A10G 24GB",  "usd_hr": 1.00, "bw_gbs": 600,  "out_tps": 1250},
    {"gpu": "A100 40GB",  "usd_hr": 1.80, "bw_gbs": 1555, "out_tps": 3100},
    {"gpu": "H100 80GB",  "usd_hr": 3.50, "bw_gbs": 3350, "out_tps": 6600},
]

def cost_per_million(usd_hr, tps):
    return (usd_hr / 3600) / max(tps, 1e-9) * 1e6

print(f"{'GPU':<13}{'$/hr':>7}{'BW GB/s':>9}{'out tok/s':>11}{'$/1M out tok':>14}{'goodput req/s*':>15}")
print("-" * 70)
rows = []
for g in GPUS:
    cpm = cost_per_million(g["usd_hr"], g["out_tps"])
    req_s = g["out_tps"] / 200          # ~200 output tokens per request
    rows.append({**g, "cpm": cpm, "req_s": req_s})
    print(f"{g['gpu']:<13}{g['usd_hr']:>7.2f}{g['bw_gbs']:>9}{g['out_tps']:>11}"
          f"{cpm:>13.3f}${req_s:>14.1f}")
print("* at ~200 output tokens/request, assuming you stay left of the knee")

print("\nSanity check - tokens/s tracks memory bandwidth, exactly as serving-fundamentals predicted:")
for r in rows:
    print(f"  {r['gpu']:<13}{r['out_tps']/r['bw_gbs']:.2f} tok/s per GB/s of bandwidth")

print("\nNow the optimizations, priced. Each multiplies effective throughput:")
BASE = rows[0]           # T4 baseline
SCENARIOS = [
    ("baseline fp16",                       1.00, "vLLM"),
    ("+ AWQ int4 weights",                  1.60, "quantization"),
    ("+ prefix caching (chat workload)",     2.10, "vLLM"),
    ("+ speculative decoding (α≈0.75, k=4)", 2.90, "speculation"),
]
print(f"\n{'scenario':<40}{'eff. tok/s':>12}{'$/1M tok':>11}{'vs base':>9}")
print("-" * 72)
for name, mult, ref in SCENARIOS:
    tps = BASE["out_tps"] * mult
    cpm = cost_per_million(BASE["usd_hr"], tps)
    print(f"{name:<40}{tps:>12.0f}{cpm:>10.3f}${cpm/rows[0]['cpm']:>8.0%}")
print("\n(multipliers are representative, not universal - measure YOUR workload with the earlier notebooks.")
print(" The point: stacked serving optimizations move cost by ~3x, which dwarfs most GPU price deltas.)")

In [ ]:
# Interactive cost explorer: pick a GPU, set traffic and optimization multiplier -> monthly bill.
JS = r'''
const ctr = root.append("div").style("font-size","13px").style("margin-bottom","8px");
function slider(label, min, max, val, step, fmt) {
  const w = ctr.append("label").style("margin-right","18px");
  w.append("span").text(label + " ");
  const out = w.append("b");
  const inp = w.append("input").attr("type","range").attr("min",min).attr("max",max)
      .attr("step",step).attr("value",val).style("vertical-align","middle").style("margin-left","6px");
  inp.on("input", () => { out.text(fmt(+inp.node().value)); draw(); });
  out.text(fmt(val));
  return inp;
}
const rps  = slider("peak traffic:", 1, 400, 60, 1, v => v + " req/s");
const mult = slider("optimization multiplier:", 1, 4, 1, 0.1, v => v.toFixed(1) + "x");
const head = slider("headroom:", 0, 100, 30, 5, v => v + "%");

const M = {top: 16, right: 200, bottom: 40, left: 108};
const iw = W - M.left - M.right, ih = H - M.top - M.bottom;
const svg = root.append("svg").attr("width", W).attr("height", H)
    .append("g").attr("transform", `translate(${M.left},${M.top})`);
const y = d3.scaleBand().domain(data.map(d => d.gpu)).range([0, ih]).padding(0.22);
const x = d3.scaleLinear().range([0, iw]);
svg.append("g").call(d3.axisLeft(y).tickSize(0)).select(".domain").remove();
const xAxis = svg.append("g").attr("transform", `translate(0,${ih})`);
svg.append("text").attr("x", iw/2).attr("y", ih+34).attr("text-anchor","middle")
   .style("font-size","12px").text("estimated monthly cost (USD, 730 hrs)");
const bars = svg.selectAll("b").data(data).join("rect")
    .attr("y", d => y(d.gpu)).attr("height", y.bandwidth()).attr("rx", 3);
const labs = svg.selectAll("t").data(data).join("text")
    .attr("y", d => y(d.gpu) + y.bandwidth()/2 + 4).style("font-size","11.5px");

function draw() {
  const traffic = +rps.node().value, m = +mult.node().value, h = 1 + (+head.node().value)/100;
  const rows = data.map(d => {
    const perGpu = (d.out_tps * m) / 200;               // ~200 output tokens per request
    const n = Math.ceil(traffic / perGpu * h);
    return {...d, n, monthly: n * d.usd_hr * 730, perGpu};
  });
  const cheapest = d3.min(rows, r => r.monthly);
  x.domain([0, d3.max(rows, r => r.monthly) * 1.35]);
  xAxis.call(d3.axisBottom(x).ticks(6, "$~s"));
  bars.data(rows).transition().duration(200)
      .attr("width", r => Math.max(2, x(r.monthly)))
      .attr("fill", r => r.monthly === cheapest ? "#2e7d32" : "#90a4ae");
  labs.data(rows).transition().duration(200)
      .attr("x", r => x(r.monthly) + 8)
      .text(r => `${r.n} GPU${r.n>1?"s":""} · $${d3.format(",.0f")(r.monthly)}/mo · ` +
                 `$${(( r.usd_hr/3600)/(r.out_tps*m)*1e6).toFixed(3)}/1M tok` +
                 (r.monthly === cheapest ? "  ← cheapest" : ""));
}
draw();
'''
show_d3(JS, rows, height=300)

**Play with the optimization multiplier.** Sliding it 1.0 → 2.5 typically cuts the monthly bill
more than switching GPU tiers does — and it costs you a config change plus a re-benchmark, not a
migration. That is the entire economic argument for this track.

Also note how the cheapest GPU changes with traffic: at low load, cheap small GPUs win (you need
one either way); at high load, the big-bandwidth cards win on $/token. Capacity planning is a
crossover problem, not a "best GPU" problem.

## Part 6 · The decision tree

You have a symptom. Which notebook do you open?

```
                        ┌──────────────────────────────────┐
                        │  What hurts?                     │
                        └──────────────┬───────────────────┘
        ┌──────────────────────────────┼──────────────────────────────┐
        ▼                              ▼                              ▼
  TTFT too high                 TPOT too slow                 Cost too high
        │                              │                              │
  ┌─────┴─────┐              ┌─────────┴─────────┐          ┌─────────┴─────────┐
  ▼           ▼              ▼                   ▼          ▼                   ▼
Waiting>0?  prompts       batch=1 idle       batch large   weights big     prompts repeat
  │          huge?        (interactive)      (saturated)      │                 │
  │           │               │                   │           ▼                 ▼
  ▼           ▼               ▼                   ▼      quantize AWQ     prefix caching
KV<70%?   chunked        speculative        you're at    (quantization)   (vLLM)
  │        prefill       decoding           the roofline
  ├─ yes → raise         (speculation)      → add replicas / bigger GPU
  │        --max-num-seqs                     (benchmarking, this one)
  └─ no  → KV-bound: shrink --max-model-len,
           quantize KV (fp8), or add replicas (quantization, logs)
```

And the order to try things, by *effort per unit of win*:

| Rank | Change | Typical win | Effort |
|---|---|---|---|
| 1 | Move shared text to the prompt's front (prefix caching) | 1.5–4× on repetitive traffic | one line |
| 2 | Right-size `--max-model-len` | more KV → more concurrency | one flag |
| 3 | AWQ/GPTQ int4 weights | ~4× weight memory, faster decode | one checkpoint swap |
| 4 | Tune `--max-num-seqs` to the knee | keeps you left of the cliff | one flag + this notebook |
| 5 | Speculative / n-gram decoding | 1.5–3× at low-to-mid load | config + acceptance monitoring |
| 6 | FP8 KV cache (Ada/Hopper) | ~2× concurrent conversations | one flag, needs newer GPU |
| 7 | More replicas / bigger GPU | linear | money |

**Do them in that order.** Most teams start at 7.

### Optional: run this load generator against a real server (needs a T4)

In [ ]:
# GPU-ONLY: identical open-loop methodology, real HTTP, real engine.
import torch
if not torch.cuda.is_available():
    print("No GPU - skipping. The simulator above already produced the full analysis.")
else:
    import subprocess, urllib.request, time, threading, os
    MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
    server = subprocess.Popen(
        ["vllm", "serve", MODEL, "--dtype", "half", "--max-model-len", "2048",
         "--gpu-memory-utilization", "0.85", "--max-num-seqs", "32", "--port", "8000"],
        stdout=open("vllm_bench.log", "w"), stderr=subprocess.STDOUT)
    for _ in range(180):
        try: urllib.request.urlopen("http://localhost:8000/health", timeout=2); break
        except Exception: time.sleep(2)

    from openai import OpenAI
    client = OpenAI(base_url="http://localhost:8000/v1", api_key="x")
    results, lock = [], threading.Lock()

    def one(i):
        t0 = time.perf_counter(); first = None; n = 0
        stream = client.chat.completions.create(
            model=MODEL, max_tokens=128, temperature=0.8, stream=True,
            messages=[{"role": "user", "content": f"Explain idea {i} in about 90 words."}])
        for ch in stream:
            if ch.choices and ch.choices[0].delta.content:
                if first is None: first = time.perf_counter() - t0
                n += 1
        total = time.perf_counter() - t0
        with lock:
            results.append({"ttft": first or total, "tpot": (total - (first or 0)) / max(n-1, 1),
                            "e2e": total, "tokens": n})

    def open_loop(rate, seconds=45):
        '''True open loop: fire on a Poisson schedule regardless of server state.'''
        results.clear(); threads = []; rng = random.Random(0); t_end = time.time() + seconds; i = 0
        while time.time() < t_end:
            time.sleep(rng.expovariate(rate))          # exponential inter-arrival = Poisson process
            t = threading.Thread(target=one, args=(i,), daemon=True); t.start()
            threads.append(t); i += 1
        for t in threads: t.join(timeout=120)
        return list(results)

    print(f"{'λ req/s':>8}{'done':>7}{'TTFT p50':>10}{'TTFT p95':>10}{'TPOT p95':>10}{'SLO met':>9}")
    for lam in (2, 5, 10, 16):
        recs = open_loop(lam)
        ok = [r for r in recs if r["ttft"] <= SLO_TTFT and r["tpot"] <= SLO_TPOT]
        print(f"{lam:>8}{len(recs):>7}{pct([r['ttft'] for r in recs], .5)*1000:>9.0f}ms"
              f"{pct([r['ttft'] for r in recs], .95)*1000:>9.0f}ms"
              f"{pct([r['tpot'] for r in recs], .95)*1000:>9.1f}ms"
              f"{len(ok)/max(len(recs),1):>8.0%}")
    print("\nCompare the shape to the simulated sweep - the knee should appear in the same way.")
    server.terminate(); server.wait(timeout=20); print("server stopped")

## Recap — the capstone checklist

Before you call a serving deployment "done":

- [ ] **Read the startup log** — you know your KV pool and concurrency ceiling ([Reading the Logs](./Serving_Logs_Observability.ipynb))
- [ ] **Benchmarked open-loop** with *your* prompt/output length distribution, warmup discarded
- [ ] **Reported p95/p99**, never means
- [ ] **Found the knee** and set `--max-num-seqs` / replica count to stay left of it
- [ ] **Defined an SLO** and measured **goodput**, not throughput
- [ ] **Dashboards alert on KV usage and queue depth** (causes), TTFT paging on top (symptom)
- [ ] **Worked the cheap optimizations first** (prefix layout → context size → quantization → speculation)
- [ ] **Computed $/1M tokens** so the next optimization has a business case

### The whole serving arc

| # | Notebook | The one thing |
|---|---|---|
| 21 | [Serving Fundamentals](./Serving_Fundamentals_KV_Cache_Batching.ipynb) | Decode is memory-bound; KV memory is the scarce resource |
| 22 | [vLLM High-Throughput Serving](./vLLM_High_Throughput_Serving.ipynb) | Paging + continuous batching = the modern engine |
| 23 | [Quantized Serving Showdown](./Quantized_Serving_Showdown.ipynb) | Fewer weight bytes = faster decode + more KV |
| 24 | [Speculative Decoding](./Speculative_Decoding_Advanced_Serving.ipynb) | Spend idle compute guessing; losslessly |
| 25 | [Serving Internals Visualized](./Serving_Internals_Visualized_D3.ipynb) | See the scheduler and the block pool move |
| 26 | [Reading the Logs](./Serving_Logs_Observability.ipynb) | Alert on causes (KV, queue), not symptoms (latency) |
| 27 | **this one** | Goodput and $/1M tokens are the only numbers that decide anything |

### Further reading
- [vLLM's own benchmark suite](https://github.com/vllm-project/vllm/tree/main/benchmarks) — `benchmark_serving.py` is the industrial-strength version of Part 2
- [Little's Law](https://en.wikipedia.org/wiki/Little%27s_law) · [The USE Method](https://www.brendangregg.com/usemethod.html) (Brendan Gregg) — capacity thinking that predates LLMs and still applies
- [DistServe](https://arxiv.org/abs/2401.09670) — where "goodput per GPU" as the objective comes from

🏁 **You've finished the serving arc — and the series.** [Back to the learning path](README.md).